# Building an Agent Team

**Mode:** Offline  
**Level:** Capstone  
**Estimated time:** 45 minutes

This notebook is meant to be read and run from top to bottom. Each code cell
changes one small part of the system, and the inspection cells show the real
Praval objects produced by that change.


## Problem and outcome

A small editorial team must turn three candidate source notes into one publishable brief. The completed system uses specialist identity, Reef choreography, a shared quality tool, project memory, and a visible message trail. It is deterministic and runs without a model provider.


In [ ]:
from pathlib import Path
import sys

candidates = [
    Path.cwd(), Path.cwd().parent,
    Path.cwd() / "examples" / "notebooks",
]
support_dir = next(path for path in candidates if (path / "support.py").is_file())
if str(support_dir) not in sys.path:
    sys.path.insert(0, str(support_dir))

from support import (
    get_events, require_env, setup_case_study, show_callout,
    show_flow, show_json, show_roles, show_spore, show_timeline, stage,
)

setup_case_study('Building an Agent Team')


## Course prerequisites

This capstone assumes: `course-01-hello-world`, `course-02-research-pipeline`, `course-04-parallel-agents`, `course-05-tool-use`, `course-06-agent-memory`. The implementation focuses on system design instead of explaining routine Python syntax.


## Architecture and responsibilities

A coordinator places a project request on the Reef. A researcher evaluates sources through a tool and records the selected evidence in project memory. An editor recalls that evidence and creates a bounded brief. A publisher captures the terminal artifact.


In [ ]:
show_roles([
('Coordinator', 'Create the bounded project request', 'human'),
('Researcher', 'Evaluate and remember evidence', 'agent'),
('Editor', 'Turn evidence into a brief', 'agent'),
('Publisher', 'Capture the final artifact', 'agent')
])


## Design decisions

- Source scoring is deterministic application code, not model judgment.
- Project memory is owned by a named Agent and stores only the selected evidence.
- Specialists communicate through typed Spores rather than direct calls.
- The publisher defines a single inspectable terminal outcome.


## Implementation

The next cells are grouped by subsystem. Each group exposes the Praval objects that matter to the architecture.


### Tool and project memory

The first subsystem owns quality policy and durable project context.


In [ ]:
from praval import Agent, agent, broadcast, get_reef, start_agents
from praval.core.reef import reset_reef
from praval.core.tool_registry import reset_tool_registry
from praval.tools import tool

reset_reef()
reset_tool_registry()
message_trail = []
final_briefs = []


@tool(
    "case_source_quality", description="Classify a bounded source score.",
    category="editorial", shared=True,
)
def source_quality(score: float) -> str:
    if not 0 <= score <= 1:
        raise ValueError("score must be between 0 and 1")
    return "strong" if score >= 0.8 else "supporting"


project_memory = Agent(
    "case-editorial-memory", provider="notebook-lifecycle",
    model="notebook-lifecycle-model", memory_enabled=True,
    memory_config={"backend": "memory"},
)
assert project_memory.memory is not None


### Research subsystem

The researcher selects evidence and emits a self-contained packet.


In [ ]:
@agent(
    "case-researcher", responds_to=["project_request"],
    tools=["case_source_quality"], auto_broadcast=False,
)
def researcher(spore):
    message_trail.append(spore)
    evaluated = [
        {**source, "quality": source_quality(source["score"])}
        for source in spore.knowledge["sources"]
    ]
    selected = [item for item in evaluated if item["quality"] == "strong"]
    for item in selected:
        project_memory.remember(
            f"Selected evidence: {item['claim']}",
            importance=item["score"], memory_type="semantic",
        )
    stage("researcher", "sources evaluated", f"{len(selected)} selected", spore)
    broadcast({
        "type": "research_packet", "project_id": spore.knowledge["project_id"],
        "selected": selected,
    })


### Editorial and publication subsystems

The editor uses the packet plus memory evidence. The publisher owns completion.


In [ ]:
@agent("case-editor", responds_to=["research_packet"], auto_broadcast=False)
def editor(spore):
    message_trail.append(spore)
    recalled = project_memory.recall("selected evidence", limit=5)
    claims = [item["claim"] for item in spore.knowledge["selected"]]
    brief = {
        "title": "Why visible agent handoffs matter",
        "claims": claims,
        "memory_evidence": [entry.content for entry in recalled],
    }
    stage("editor", "brief assembled", f"{len(claims)} claims", spore)
    broadcast({
        "type": "final_brief", "project_id": spore.knowledge["project_id"],
        "brief": brief,
    })


@agent("case-publisher", responds_to=["final_brief"], auto_broadcast=False)
def publisher(spore):
    message_trail.append(spore)
    final_briefs.append(spore.knowledge)
    stage("publisher", "artifact accepted", spore.knowledge["project_id"], spore)


## End-to-end run

One project request now drives the complete editorial team.


In [ ]:
sources = [
    {"claim": "Spores make handoffs inspectable.", "score": 0.95},
    {"claim": "Completion waits include cascading work.", "score": 0.9},
    {"claim": "Every workflow needs ten agents.", "score": 0.3},
]
start_agents(
    researcher, editor, publisher,
    initial_data={
        "type": "project_request", "project_id": "editorial-001",
        "sources": sources,
    },
    channel="case-editorial-team",
)
reef = get_reef()
assert reef.wait_for_completion(timeout=10)
assert len(final_briefs) == 1
assert len(final_briefs[0]["brief"]["claims"]) == 2
assert len(message_trail) == 3


## Inspect the system

Inspect the evidence left by the completed run.


In [ ]:
show_json(final_briefs[0], "Published brief")
for index, spore in enumerate(message_trail, start=1):
    show_spore(spore, f"Message {index}: {spore.knowledge['type']}")
show_json(project_memory.memory.get_memory_stats(), "Project memory state")
show_timeline()


## Failure modes and tradeoffs

- If source scoring becomes model-based, quality becomes probabilistic and needs evaluation evidence.
- A shared memory Agent is simple here but needs access policy in a multi-tenant deployment.
- The editor currently requires at least one strong claim; production code needs an explicit no-evidence outcome.
- In-memory Reef and memory are process-local; distributed and persistent backends add operational cost.


## Extensions

- Add a fact-checker between editor and publisher and preserve the message trail.
- Run two project IDs concurrently and prove memory and aggregation remain scoped.
- Replace source scoring with a reviewed rubric tool while keeping the agent contracts unchanged.


## Cleanup

Release project resources explicitly.


In [ ]:
project_memory.memory.clear_agent_memories(project_memory.name)
project_memory.close()
for function in (researcher, editor, publisher):
    function._praval_agent.close()
assert reef.shutdown(timeout=10)
reset_tool_registry()
